# Shadow Slave full-novel index rebuild (Colab) - fp32 + int8

Rebuilds the bge-large retrieval index for **all 3,160 chapters** (`novel_chunks`
+ `notebook_statements`) and zips both for Weaver. Plan 018.5 slice 6.

**Before you start (once per runtime):**
1. Runtime -> Change runtime type -> **T4 GPU** -> Save.
2. Upload **`shadow-slave.zip`** and **`knowledge.zip`** to the file pane (left folder icon).
3. Run cells **in order**. On a re-run after a runtime death, cells 4-7 can be
   skipped if `/content/index-fp32` and `/content/int8-model` already exist.


In [ ]:
# 1. Install dependencies with ONLY the GPU onnxruntime (CUDA 12 build).
#    Colab's T4 is CUDA 12; the default onnxruntime-gpu (1.28+) is a CUDA 13 build and
#    fails to load. The CUDA 12 build lives on Microsoft's feed.
!pip uninstall -y onnxruntime onnxruntime-gpu 2>/dev/null | tail -1
!pip install -q fastembed qdrant-client onnx tokenizers
!pip install -q onnxruntime-gpu --extra-index-url https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/
!python -c "import fastembed, qdrant_client, onnxruntime as ort; print('IMPORTS OK:', ort.__version__, ort.get_available_providers())"
!python -c "import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - Runtime > Change runtime type > T4 GPU')"


In [ ]:
# 2. Unzip the novel and notebook, sanity-check the layout.
!unzip -q -o shadow-slave.zip
!unzip -q -o knowledge.zip
!ls novels/shadow-slave | tail -2
!ls knowledge/shadow-slave/reading | tail -2    # expect 3160.json
!ls knowledge/shadow-slave/reading | wc -l      # expect 3160


In [ ]:
# 3. Fetch the LATEST runner + weaver modules from GitHub (no re-uploads, ever).
!mkdir -p weaver/retrieval
!curl -sfL https://raw.githubusercontent.com/haxsysgit/Weaver/main/scripts/colab_build_runner.py -o colab_build_runner.py
!curl -sfL https://raw.githubusercontent.com/haxsysgit/Weaver/main/src/weaver/retrieval/chunker.py -o weaver/retrieval/chunker.py
!curl -sfL https://raw.githubusercontent.com/haxsysgit/Weaver/main/src/weaver/retrieval/experiment.py -o weaver/retrieval/experiment.py
!wc -l colab_build_runner.py weaver/retrieval/chunker.py weaver/retrieval/experiment.py
!python -c "import sys; sys.path.insert(0, '.'); from weaver.retrieval.chunker import chunk_chapter; from weaver.retrieval.experiment import create_novel_collection, create_notebook_collection; print('RUNNER IMPORTS OK')"


## 4. Build 1 of 2: fp32 index (the baseline)

Runs the real weaver runner (real chunker, real collection creators) on CUDA.
Expect **~15-30 minutes** on a working T4. If CUDA fails the runner falls back
to CPU automatically (slower, correct). Progress prints every 500 chunks.


In [ ]:
# 4. fp32 build (CUDA on T4) - single line, no continuation escapes
!python colab_build_runner.py --novel-dir novels/shadow-slave --notebook-dir knowledge/shadow-slave --out /content/index-fp32 --ceiling 3160
!du -sh /content/index-fp32


## 5. Self-quantize fp32 -> int8

`quantize_dynamic` on our trusted fp32 onnx: MatMul weights only (the
512-token position-table fix). ~30s, 1.3GB -> ~430MB. The runner needs
`tokenizer.json` beside the onnx, so we copy it from the fastembed cache.


In [ ]:
# 5. Quantize to int8 (MatMul weights only)
import glob, os, shutil, tempfile
from onnxruntime.quantization import quantize_dynamic, QuantType

# Locate the fp32 bge-large onnx wherever fastembed cached it on this box.
# Local dev sets FASTEMBED_CACHE_PATH; Colab defaults to /tmp/fastembed_cache.
cache_dir = os.getenv("FASTEMBED_CACHE_PATH", os.path.join(tempfile.gettempdir(), "fastembed_cache"))
matches = glob.glob(os.path.join(cache_dir, "**", "model.onnx"), recursive=True)
assert matches, f"no model.onnx under {cache_dir}; ls: {os.listdir(cache_dir) if os.path.isdir(cache_dir) else 'missing'}"
# bge-large fp32 is ~1.3GB; the sparse bm42 model is tiny - pick by size.
fp32 = max(matches, key=os.path.getsize)
tok = os.path.join(os.path.dirname(fp32), "tokenizer.json")
print("found fp32 model:", fp32, f"{os.path.getsize(fp32)/1e6:.0f} MB")
os.makedirs("/content/int8-model", exist_ok=True)
quantize_dynamic(
    fp32,
    "/content/int8-model/model_quantized.onnx",
    weight_type=QuantType.QInt8,
    op_types_to_quantize=["MatMul"],
)
shutil.copy(tok, "/content/int8-model/tokenizer.json")
print("int8 model:", round(os.path.getsize("/content/int8-model/model_quantized.onnx")/1e6), "MB")


## 6. Build 2 of 2: int8 index (the v1 target)

Same runner, `--dense` points at the local int8 onnx, **`--provider cpu`**:
dynamic-quantized models dequantize per layer on GPU (memcpy crawl), so CPU
is the fast path here. Progress prints every 500 chunks. Expect **~15 minutes**.


In [ ]:
# 6. int8 build (CPU) - single line, no continuation escapes
!python colab_build_runner.py --novel-dir novels/shadow-slave --notebook-dir knowledge/shadow-slave --out /content/index-int8 --ceiling 3160 --dense /content/int8-model/model_quantized.onnx --provider cpu
!du -sh /content/index-int8


In [ ]:
# 7. Verify both indexes BEFORE downloading (proves they are real, not partial)
from qdrant_client import QdrantClient
for name, path in [("fp32", "/content/index-fp32"), ("int8", "/content/index-int8")]:
    try:
        c = QdrantClient(path=path)
        for col in c.get_collections().collections:
            print(name, col.name, c.count(col.name).count, "points")
        c.close()
    except Exception as e:
        print(name, "ERROR:", e)
# Expect: novel_chunks ~6295, notebook_statements ~5341 for BOTH indexes


## 7. Download both indexes to your machine

Save both zips somewhere you can find (Downloads folder is fine). Tell the
assistant where they landed - the swap into `.weaver/retrieval/index` happens
back on your machine.


In [ ]:
# 8. Zip and download
!cd /content && zip -q -r index-fp32.zip index-fp32
!cd /content && zip -q -r index-int8.zip index-int8
!ls -lh /content/index-fp32.zip /content/index-int8.zip
from google.colab import files
files.download('/content/index-fp32.zip')
files.download('/content/index-int8.zip')


## Back on your machine

The assistant handles the swap into `.weaver/retrieval/index` - you only
need to download the zips and say where they landed.
